# 10 — Advanced Feature Engineering (The Santander Special)
This notebook implements the "secret sauce" features that helped winners in the original competition:
1. **Row-wise Statistics**: Counting zeros and non-zeros.
2. **Feature Interactions**: Combining the most powerful features into new terms.

**Outputs:**
1. `pickles/train_advanced.pkl`
2. `pickles/test_advanced.pkl`

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set your project path on Drive
DRIVE_PATH = '/content/drive/MyDrive/santander-customer-satisfaction/'
PICKLE_DIR = DRIVE_PATH + 'pickles/'

import pandas as pd
import numpy as np

print('Setup complete.')

Mounted at /content/drive
Setup complete.


## 1. Load Current Clean Data

In [2]:
train = pd.read_pickle(f'{PICKLE_DIR}train_clean.pkl')
test  = pd.read_pickle(f'{PICKLE_DIR}test_clean.pkl')

print(f"Initial Train shape: {train.shape}")
print(f"Initial Test shape: {test.shape}")

Initial Train shape: (76020, 97)
Initial Test shape: (75818, 96)


## 2. Row-wise Statistics (Sparsity Features)
In Santander, the number of zero-valued features is a strong proxy for customer activity level.

In [3]:
# Define features to calculate stats over (exclude ID and TARGET)
features = [c for c in train.columns if c not in ['ID', 'TARGET']]

def add_row_stats(df):
    # Number of zeros per row
    df['n_zeros'] = (df[features] == 0).sum(axis=1)
    # Number of non-zeros per row
    df['n_non_zeros'] = (df[features] != 0).sum(axis=1)
    # Row-wise mean and std (can sometimes catch outliers)
    df['row_mean'] = df[features].mean(axis=1)
    df['row_std'] = df[features].std(axis=1)
    return df

print("Adding row-wise statistics...")
train = add_row_stats(train)
test = add_row_stats(test)

print(f"New Train shape: {train.shape}")

Adding row-wise statistics...
New Train shape: (76020, 101)


## 3. Top Feature Interactions
Based on our previous LightGBM gain analysis, the most important features are often `var15` (Age), `var38` (Mortgage/Value), and `saldo_var30`. Let's create interactions for these.

In [4]:
top_features = ['var15', 'var38', 'saldo_var30', 'saldo_medio_var5_hace2']

def add_interactions(df):
    # Interaction 1: Age * Mortgage Value
    df['var15_x_var38'] = df['var15'] * df['var38']

    # Interaction 2: Age / (Balance + 1) -> Balance per year of life
    df['var15_div_saldo'] = df['var15'] / (df['saldo_var30'] + 1)

    # Interaction 3: var38 / (Age + 1)
    df['var38_div_var15'] = df['var38'] / (df['var15'] + 1)

    return df

print("Adding top interactions...")
train = add_interactions(train)
test = add_interactions(test)

print(f"Final Advanced Train shape: {train.shape}")

Adding top interactions...
Final Advanced Train shape: (76020, 104)


## 4. Save Advanced Datasets

In [5]:
train.to_pickle(f'{PICKLE_DIR}train_advanced.pkl')
test.to_pickle(f'{PICKLE_DIR}test_advanced.pkl')

print("✅ Advanced Feature Datasets saved successfully!")
print("Now rerun your model notebooks using 'train_advanced.pkl' for a potential boost!")

✅ Advanced Feature Datasets saved successfully!
Now rerun your model notebooks using 'train_advanced.pkl' for a potential boost!
